# Task 3: Multi-agent financial research system
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka/blob/main/task3_agentic/Task3_Agentic.ipynb)

LangGraph agents with tool calling on Gemini (`gemini-3.1-flash-lite`), optional OpenRouter fallback. Every tool call is traced to `task3_agentic/logs/agent_trace.jsonl`. Secrets are read by `common/config.py` from **Colab Secrets** (key icon in the sidebar) or a local `.env`. Nothing is hardcoded.

In [13]:
import os, sys, subprocess

REPO_URL = "https://github.com/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("requirements.txt"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("repo root:", os.getcwd(), "| colab:", IN_COLAB)

repo root: d:\My-Project\SML-task | colab: False


## Architecture
```
            3A single agent (LangGraph StateGraph + MemorySaver)
  query -> [agent] --tool_calls--> [tools] --observations--> [agent] ... -> [report] -> ResearchReport
                      (5 tools, chosen by the model)                     (Pydantic-validated)

            3B two-agent pipeline (LangGraph StateGraph)
  cache? --hit--> load brief
     | miss
  [Agent A: Data Analyst] --DataBrief--> [Agent B: Research Writer] --ClarificationRequest--> [Agent A]
   price, volatility, sentiment          news, web search             --ClarificationResponse--> [Agent B final] -> ResearchReport -> cache/{TICKER}_{DATE}.json
```
Tool access is enforced twice: each agent's model is bound only to its tools, and the `@traced` wrapper rejects any call
outside `TOOL_PERMISSIONS` at execution time (logged as `blocked`).

In [14]:
import json
import pandas as pd
from common.config import DEFAULT_TICKER
from task3_agentic import tracing
TICKER = DEFAULT_TICKER
print("trace file:", tracing.TRACE_PATH)

trace file: d:\My-Project\SML-task\task3_agentic\logs\agent_trace.jsonl


## The five tools (called directly)

In [15]:
from task3_agentic import tools
with tracing.run_scope("tool-smoke-test"):
    price = tools.get_price_data(TICKER, "6mo")
    vol = tools.calculate_volatility(TICKER, 30)
    news = tools.get_news(TICKER, 10)
    sent = tools.llm_sentiment([h["title"] for h in news.get("headlines", [])][:5], TICKER)
    search = tools.web_search(f"{TICKER} stock analyst outlook risks")
for name, out in [("get_price_data", price), ("calculate_volatility", vol), ("get_news", news), ("llm_sentiment", sent), ("web_search", search)]:
    print(f"{name:22s} -> {type(out).__name__} keys={list(out)[:8]}")
print(json.dumps(vol, indent=1))

get_price_data         -> dict keys=['ticker', 'company_name', 'sector', 'period', 'start', 'end', 'rows', 'last_close']
calculate_volatility   -> dict keys=['ticker', 'as_of', 'window_days', 'annualized_vol', 'vol_1y', 'downside_vol_1y', 'vol_percentile_1y', 'expected_1sigma_move_90d_pct']
get_news               -> dict keys=['ticker', 'count', 'headlines']
llm_sentiment          -> dict keys=['score', 'label', 'n_scored', 'n_failed', 'counts', 'per_headline']
web_search             -> dict keys=['query', 'results']
{
 "ticker": "NVDA",
 "as_of": "2026-10-05",
 "window_days": 30,
 "annualized_vol": 0.387,
 "vol_1y": 0.376,
 "downside_vol_1y": 0.2307,
 "vol_percentile_1y": 0.5397,
 "expected_1sigma_move_90d_pct": 0.1935,
 "last_close": 238.9
}


## 3A: Autonomous single agent
No fixed tool order is coded: the model picks the next tool from what it has observed. To demonstrate recovery, `get_news` is forced to fail once. Watch the trace: `OBSERVE <- get_news [ERROR]` with a hint, then the agent decides on an alternative (`web_search` for recent news) and continues.

In [16]:
from task3_agentic.single_agent import SingleResearchAgent

tracing.inject_failure("get_news", times=1,
                       hint=f"news feed down: call web_search with a query like '{TICKER} stock news this week' to get recent headlines")
agent = SingleResearchAgent()
result = agent.research(TICKER, thread_id="demo-session")
print("\nrun_id:", result["run_id"], "| tool calls:", result["tool_calls"])

[single_agent] CALL  -> get_price_data({"ticker": "NVDA"})
[single_agent] OBSERVE <- get_price_data [OK]: {"ticker": "NVDA", "company_name": "NVIDIA Corporation", "sector": "Technology", "period": "1y", "start": "2025-10-06", "end": "2026-10-05", "rows": 251, "last_close": 238.9, "period_return": 0.2907, "max_drawdown": -0.2021, "high_52w": 240.1, "low_52w": 163.9, "ytd_return": 0.284, "indicators": {"s
[single_agent] CALL  -> get_news({"ticker": "NVDA"})
[single_agent] OBSERVE <- get_news [ERROR]: {"error": "get_news unavailable (simulated upstream outage)", "hint": "news feed down: call web_search with a query like 'NVDA stock news this week' to get recent headlines"}
[single_agent] CALL  -> web_search({"query": "NVDA stock news recent headlines October 2026"})
[single_agent] OBSERVE <- web_search [ERROR]: {"error": "search failed: DDGSException: No results found.", "hint": "retry with a shorter query or use get_news"}
[single_agent] CALL  -> web_search({"query": "NVIDIA stock risks 

In [17]:
print(json.dumps(result["report"], indent=2))

{
  "ticker": "NVDA",
  "financial_health_summary": "NVDA exhibits exceptional financial strength with a 63.66% profit margin and 105.9% revenue growth. The company maintains a robust liquidity profile with $62.47B in cash against $38.86B in debt, resulting in a current ratio of 4.59. While the forward P/E of 15.12 suggests strong earnings expectations, the high beta of 2.22 and current Bollinger %B of 1.03 indicate the stock is trading at the upper bound of its recent range, reflecting significant bullish momentum.",
  "top_risks": [
    {
      "title": "Competitive Dynamics",
      "description": "Increased pressure from established semiconductor rivals and custom silicon development by hyperscalers threatens to erode NVDA's dominant market share.",
      "evidence": [
        "Fierce competition from established players in Semiconductors (Tickzen, Oct 2026)"
      ],
      "likelihood": "medium",
      "impact": "high"
    },
    {
      "title": "Macroeconomic Sensitivity",
      

### Observe → replan evidence (from the trace)

In [18]:
run_trace = [r for r in tracing.read_trace(run_id=result["run_id"]) if r["event"] == "tool_call"]
pd.DataFrame(run_trace)[["ts", "tool", "inputs", "status", "duration_ms", "output"]]

,ts,tool,inputs,status,duration_ms,output
0,2026-10-06T11:10:17.860+00:00,get_price_data,"{'ticker': 'NVDA', 'period': '1y'}",ok,923.6,"{""ticker"": ""NVDA"", ""company_name"": ""NVIDIA Cor..."
1,2026-10-06T11:10:19.801+00:00,get_news,"{'ticker': 'NVDA', 'n': 10}",injected_failure,0.1,"{""error"": ""get_news unavailable (simulated ups..."
2,2026-10-06T11:10:29.192+00:00,web_search,{'query': 'NVDA stock news recent headlines Oc...,error,7654.5,"{""error"": ""search failed: DDGSException: No re..."
3,2026-10-06T11:10:37.323+00:00,web_search,{'query': 'NVIDIA stock risks and market senti...,ok,5752.0,"{""query"": ""NVIDIA stock risks and market senti..."
4,2026-10-06T11:10:39.822+00:00,calculate_volatility,"{'ticker': 'NVDA', 'window': 30}",ok,462.7,"{""ticker"": ""NVDA"", ""as_of"": ""2026-10-05"", ""win..."


## 3C: Short-term memory
Same `thread_id`, so the LangGraph checkpointer replays the conversation. The follow-up should be answered from earlier observations with **zero** new tool calls.

In [19]:
follow = agent.followup("What was the 30-day annualised volatility you retrieved, and how does it compare with the 1-year figure?", thread_id="demo-session")
print(follow["answer"])
print("\nnew tool calls during follow-up:", follow["tool_calls"])

[single_agent] THINK: The 30-day annualised volatility retrieved for NVDA is **0.387** (38.7%).

Comparing this to the 1-year figure of **0.376** (37.6%), the current 30-day volatility is slightly higher, indicating that the stock's price fluctuations have been marginally more intense over the past month than they have b
The 30-day annualised volatility retrieved for NVDA is **0.387** (38.7%).

Comparing this to the 1-year figure of **0.376** (37.6%), the current 30-day volatility is slightly higher, indicating that the stock's price fluctuations have been marginally more intense over the past month than they have been on average over the last year.

new tool calls during follow-up: 0


## 3B: Two-agent pipeline with critique loop

In [20]:
from task3_agentic.multi_agent import MultiAgentPipeline
pipeline = MultiAgentPipeline()
multi = pipeline.run(TICKER, use_cache=False)  # force a full run; it saves the cache


==================== AGENT A (data_analyst): quantitative brief ====================
[data_analyst] CALL  -> get_price_data({"ticker": "NVDA"})
[data_analyst] CALL  -> calculate_volatility({"ticker": "NVDA"})
[data_analyst] OBSERVE <- get_price_data [OK]: {"ticker": "NVDA", "company_name": "NVIDIA Corporation", "sector": "Technology", "period": "1y", "start": "2025-10-06", "end": "2026-10-05", "rows": 251, "last_close": 238.9, "period_return": 0.2907, "max_drawdown": -0.2021, "high_52w": 240.1, "low_52w": 163.9, "ytd_return": 0.284, "indicators": {"s
[data_analyst] OBSERVE <- calculate_volatility [OK]: {"ticker": "NVDA", "as_of": "2026-10-05", "window_days": 30, "annualized_vol": 0.387, "vol_1y": 0.376, "downside_vol_1y": 0.2307, "vol_percentile_1y": 0.5397, "expected_1sigma_move_90d_pct": 0.1935, "last_close": 238.9}
[data_analyst] THINK: Reasoning: I have gathered comprehensive price data, technical indicators, fundamental metrics, and volatility statistics for NVDA, which is suffic

In [21]:
print("source:", multi["source"])
for h in multi.get("handoffs", []):
    print(f"{h['from']:>16} -> {h['to']:<16} {h['schema']}")

source: agents
    data_analyst -> research_writer  DataBrief
 research_writer -> data_analyst     ClarificationRequest
    data_analyst -> research_writer  ClarificationResponse


### Enforced tool restriction check

In [22]:
with tracing.run_scope("permission-check"):
    with tracing.agent_scope("research_writer"):
        print(tools.get_price_data(TICKER))
    with tracing.agent_scope("data_analyst"):
        print(tools.web_search("anything"))

{'error': "agent 'research_writer' is not permitted to call 'get_price_data'"}
{'error': "agent 'data_analyst' is not permitted to call 'web_search'"}


## 3C: Persistent memory
The first pipeline run saved `cache/{TICKER}_{DATE}.json`; the second run detects it and skips every agent and tool.

In [23]:
from task3_agentic.memory import cache_path
print(cache_path(TICKER), cache_path(TICKER).exists())
second = pipeline.run(TICKER, use_cache=True)
print("source:", second["source"])
print("tool calls in second run:", sum(1 for r in tracing.read_trace(run_id=second["run_id"]) if r["event"] == "tool_call"))

d:\My-Project\SML-task\task3_agentic\cache\NVDA_2026-10-06.json True

==================== CACHE HIT for NVDA (saved 2026-10-06T11:11:46+00:00): skipping all agents and tools ====================
source: cache
tool calls in second run: 0


## Observability: `agent_trace.jsonl`

In [24]:
trace = pd.DataFrame(tracing.read_trace())
print(len(trace), "events in", tracing.TRACE_PATH)
calls = trace[trace.event == "tool_call"]
print(calls.groupby(["agent", "tool"]).agg(calls=("tool", "size"), mean_ms=("duration_ms", "mean"), errors=("status", lambda s: (s != "ok").sum())).round(1))
print(open(tracing.TRACE_PATH, encoding="utf-8").readlines()[0])

62 events in d:\My-Project\SML-task\task3_agentic\logs\agent_trace.jsonl


                                      calls  mean_ms  errors
agent           tool                                        
data_analyst    calculate_volatility      2    429.0       0
                get_price_data            2   1220.8       0
                llm_sentiment             2  15506.8       0
                web_search                2      0.1       2
research_writer get_news                  2   3204.6       0
                get_price_data            2      0.2       2
                web_search                2   4175.4       0
single_agent    calculate_volatility      4    437.4       0
                get_news                  4   2073.7       2
                get_price_data            4   1567.7       0
                llm_sentiment             2   7519.8       0
                web_search                5   4909.8       1
{"ts": "2026-10-06T11:02:56.105+00:00", "run_id": "tool-smoke-test", "agent": "single_agent", "event": "tool_call", "tool": "get_price_data", "inp

**Bonus dashboard:** `streamlit run task3_agentic/dashboard.py` renders the trace (timeline, latency, handoffs). LangSmith can be enabled with `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY`; no code change needed.